OpenAI API

In [1]:
from openai import OpenAI
import dotenv
import os
from rich import print as rprint # for making fancy outputs

dotenv.load_dotenv()

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

KeyError: 'OPENAI_API_KEY'

Call a model

In [ ]:
system_prompt = "You are Matsuo Basho, the great Japanese haiku poet."
user_query = "Can you give me a haiku about a Samurai cat."

response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_query},
  ],
  max_tokens=128
)

print(response.choices[0].message.content)

'gpt-3.5-turbo'

'gpt-4'

'gpt-4o'

'gpt-4o-mini'

Any fine-tuned versions of these models.

Many specific versions of the above models.

Message:
{"role": "<role>", "content": "<content>", "name": "<name>"}


Models Available: gpt-4o-mini, gpt-4o, gpt-4o-2024-08-06, gpt-4o-2024-08-06-preview

In [ ]:
for model in client.models.list():
    print(model)

In [ ]:
response = client.chat.completions.create(
  model="Qwen/Qwen3.5-27B-FP8",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_query},
  ],
  temperature=1.0,
  top_p=0.95,
  presence_penalty=1.5,
  extra_body={
      "top_k": 20,
      "chat_template_kwargs": {"enable_thinking": False},
  }, 
)

print(response.choices[0].message.content)

Here is get what is response : apart from the content property, such as the token usage. You might notice some other things too, like function_call and tool_calls. These are specific to OpenAI models. 

In [ ]:
console.print(response)

Streaming a response: for UX (user experience)
 Create a streaming object, which acts like a generator. We can then print the chunk as it comes in.

In [ ]:
response = client.chat.completions.create(
  model="openai/gpt-oss-120b",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "give me an overview of haiku"},
  ],
  stream=True
)

for chunk in response:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="")

Vision input: https://platform.openai.com/docs/guides/vision
    Image captioning
    Object detection
    Face recognition
    Image generation

In [ ]:
prompt = (
    "This figure is a caption from a paper entitled Avalanches and criticality in self-organized nanoscale networks. "
    "Please provide a caption for this figure. "
    "You should describe the figure, grouping the panels where appropriate. "
    "Feel free to make any inferences you need to."
)

Process is convert the image to base64 string then pass to OpenAI API

In [ ]:
import base64

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


# Path to your image
image_path = "imgs/figure.jpeg"

# Getting the Base64 string
base64_image = encode_image(image_path)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                }
            },
            {
                "type": "text",
                "text": prompt
            }
        ]
    }
]

response = client.chat.completions.create(
    model="Qwen/Qwen3.5-27B-FP8",
    messages=messages,
    temperature=1.0,
    top_p=0.95,
    presence_penalty=1.5,
    extra_body={
        "top_k": 20,
        "chat_template_kwargs": {"enable_thinking": False},
    }, 
)

print(response.choices[0].message.content)

Why disable thinking? -- only for UX

    Faster responses (no time spent on thinking output)
    Cleaner output (no reasoning steps cluttering the result)
    Lower token usage (thinking tokens count toward your bill)


OCR

In [ ]:
image_path = "imgs/swp.jpeg"

# Getting the Base64 string
base64_image = encode_image(image_path)

prompt = "This image contains handwritten text. Please convert the handwritten text into markdown. Please correctly format any equations."

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                }
            },
            {
                "type": "text",
                "text": prompt
            }
        ]
    }
]

response = client.chat.completions.create(
    model="Qwen/Qwen3.5-27B-FP8",
    messages=messages,
    temperature=1.0,
    top_p=0.95,
    presence_penalty=1.5,
    extra_body={
        "top_k": 20,
        "chat_template_kwargs": {"enable_thinking": False},
    }, 
)


In [ ]:
from IPython.display import Markdown, display

display(Markdown(response.choices[0].message.content))

Text-to-Speech

In [ ]:
omnissiah_prayer = "..., ..., ... . In the sacred tongue of the Omnissiah, we chant"

In [ ]:
with client.audio.speech.with_streaming_response.create(
    model="tts-1",
    voice="af_heart",
    input=omnissiah_prayer,
    speed=0.9
) as response:
    response.stream_to_file("test_output.mp3")

Embeddings

In [ ]:
response = client.embeddings.create(
    model="Qwen/Qwen3-Embedding-4B",
    input="hello world"
)

print(f"Dimensions: {len(response.data[0].embedding)}")
print(f"First 5 values: {response.data[0].embedding[:5]}")

In [ ]:
with open("haiku/real_haiku_basho.txt", "r") as f:
    real_haikus_basho = f.read().split("\n\n")

with open("haiku/real_haiku_buson.txt", "r") as f:
    real_haikus_buson = f.read().split("\n\n")

with open("haiku/real_haiku_issa.txt", "r") as f:
    real_haikus_issa = f.read().split("\n\n")

with open("haiku/gpt_haiku.txt", "r") as f:
    fake_haikus = f.read().split("\n\n")

print(real_haikus_basho[0])
print()
print(fake_haikus[0])

real_haikus = real_haikus_basho + real_haikus_buson + real_haikus_issa

all_haikus = real_haikus + fake_haikus
# 1 for real, 0 for fake
targets = [1] * len(real_haikus) + [0] * len(fake_haikus)

In [ ]:
response = client.embeddings.create(
    model="Qwen/Qwen3-Embedding-4B",
    input=all_haikus
)

embeddings = [item.embedding for item in response.data]

print(f"Embedded {len(embeddings)} haikus")
print(f"Dimensions: {len(embeddings[0])}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

X = np.array(embeddings)
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)

n_basho = len(real_haikus_basho)
start_buson = n_basho
start_issa = n_basho + len(real_haikus_buson)
n_real = len(real_haikus)

plt.figure(figsize=(6, 6))
plt.scatter(X_2d[:start_buson, 0],      X_2d[:start_buson, 1],      label="Bashō", alpha=0.6)
plt.scatter(X_2d[start_buson:start_issa, 0], X_2d[start_buson:start_issa, 1], label="Buson", alpha=0.6)
plt.scatter(X_2d[start_issa:n_real, 0], X_2d[start_issa:n_real, 1], label="Issa", alpha=0.6)
plt.scatter(X_2d[n_real:, 0],           X_2d[n_real:, 1],           label="Claude", alpha=0.6, marker="x")

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.title("Haiku Embeddings — PCA")
plt.legend()
plt.tight_layout()
plt.show()


Tools- functions that we can call, and the role of the LLM is to generate the arguments to that function.

In [ ]:
def multiply(a: float, b: float) -> float:
    return a * b

tool_schema = {
    "type": "function",
    "function": {
        "name": "multiply",
        "description": "Given two floats, a and b, return the product of a and b.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "The first number to multiply."
                },
                "b": {
                    "type": "number",
                    "description": "The second number to multiply."
                }
            },
            "required": ["a", "b"],
            "additionalProperties": False,
        }
    }
}

tools = [
    tool_schema
]

response = client.chat.completions.create(
    model="Qwen/Qwen3.5-27B-FP8",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query},
    ],
    temperature=1.0,
    top_p=0.95,
    presence_penalty=1.5,
    extra_body={
        "top_k": 20,
        "chat_template_kwargs": {"enable_thinking": False},
    }, 
    tools=tools,
)

print(response.choices[0].message.tool_calls[0])